In [2]:
import pandas as pd
import numpy as np

In [4]:
input_file = "orders.csv"
output_file = "orders_cleaned.csv"
issues_file = "orders_issues.csv"

In [6]:
df = pd.read_csv(input_file)

In [7]:
# Standardize column names
df.columns = df.columns.str.strip().str.lower()

In [8]:
# Treat blank strings as missing
df = df.replace(r"^\s*$", np.nan, regex=True)

In [9]:
# Clean all text columns first
for col in df.select_dtypes(include="object").columns:
    df[col] = df[col].astype(str).str.strip()
    df[col] = df[col].replace("nan", np.nan)

C:\Users\makwa\AppData\Local\Temp\ipykernel_10856\2664835988.py:2: Pandas4Warning: For backward compatibility, 'str' dtypes are included by select_dtypes when 'object' dtype is specified. This behavior is deprecated and will be removed in a future version. Explicitly pass 'str' to `include` to select them, or to `exclude` to remove them and silence this warning.
See https://pandas.pydata.org/docs/user_guide/migration-3-strings.html#string-migration-select-dtypes for details on how to write code that works with pandas 2 and 3.
  for col in df.select_dtypes(include="object").columns:


In [10]:

# Keep ids as text
df["order_id"] = df["order_id"].astype(str).str.strip()
df["customer_id"] = df["customer_id"].astype(str).str.strip()

In [11]:
# Date fields
df["order_date"] = pd.to_datetime(df["order_date"], errors="coerce")
df["order_month"] = df["order_date"].dt.strftime("%Y-%m")
df["order_year"] = df["order_date"].dt.year.astype("Int64")
df["day_of_week"] = df["order_date"].dt.day_name()
df["order_quarter"] = "Q" + df["order_date"].dt.quarter.astype("Int64").astype(str)

In [12]:
# Booleans
for col in ["is_weekend", "is_month_end"]:
    df[col] = df[col].astype(str).str.strip().str.lower().map({"true": True, "false": False})

In [13]:
# City cleanup
df["city"] = (
    df["city"].astype(str).str.strip().str.lower().replace({
        "londen": "london",
        "londn": "london",
        "leeds ": "leeds",
        "leds": "leeds",
        "leedes": "leeds",
        "liverpol": "liverpool",
        "manchestor": "manchester",
        "manchestr": "manchester",
        "birmigham": "birmingham",
        "brumingham": "birmingham",
        "cardif": "cardiff",
        "glasgoe": "glasgow",
        "edinbugh": "edinburgh",
        "edinburg": "edinburgh",
        "liverpoo": "liverpool"
    }).str.title()
)

In [14]:
# Standardize category columns
df["region"] = df["region"].astype(str).str.strip().str.title()
df["acquisition_channel"] = df["acquisition_channel"].astype(str).str.strip().str.title()
df["status"] = df["status"].astype(str).str.strip().str.title()

In [15]:
# Numeric columns
for col in ["subtotal", "shipping_cost", "order_total", "discount_pct", "num_items"]:
    df[col] = pd.to_numeric(df[col], errors="coerce")

In [16]:
# Basic numeric validation
for col in ["subtotal", "shipping_cost", "order_total", "discount_pct", "num_items"]:
    df.loc[df[col] < 0, col] = np.nan

In [17]:
# Normalize discount_pct
# If values are like 25, 17, 13 etc., convert to decimal
df["discount_pct"] = np.where(df["discount_pct"] > 1, df["discount_pct"] / 100, df["discount_pct"])

In [18]:
# Calculate discount amount
df["discount_amount"] = (df["subtotal"] * df["discount_pct"]).round(2)

In [19]:
# Calculate real shipping cost using discount-adjusted total
# Formula: shipping = order_total - (subtotal - discount_amount)
df["shipping_cost_calc"] = (df["order_total"] - (df["subtotal"] - df["discount_amount"])).round(2)

In [20]:
# Check if original shipping matches calculated shipping
df["shipping_cost_check"] = np.where(
    df["shipping_cost"].notna() & (df["shipping_cost"].round(2) != df["shipping_cost_calc"].round(2)),
    "check",
    ""
)

In [21]:
# Fill missing shipping cost with calculated value
df.loc[df["shipping_cost"].isna(), "shipping_cost"] = df["shipping_cost_calc"]

In [22]:
# Recalculate order_total to be consistent
df["order_total_calc"] = (df["subtotal"] - df["discount_amount"] + df["shipping_cost"]).round(2)

In [23]:
# Check if order_total matches calculated total
df["order_total_check"] = np.where(
    df["order_total"].notna() & (df["order_total"].round(2) != df["order_total_calc"].round(2)),
    "check",
    ""
)

In [35]:
year_check = (
    df["order_date"].notna() &
    df["order_year"].notna() &
    (df["order_date"].dt.year.astype("Int64") != df["order_year"])
)

df["year_check"] = np.where(year_check, "check", "")

In [36]:
df["year_check"] = np.where(
    ((df["order_date"].dt.year.astype("Int64") != df["order_year"]).fillna(False)),
    "check",
    ""
)

In [37]:
df["month_check"] = np.where(
    (
        df["order_date"].notna() &
        df["order_month"].notna() &
        (df["order_date"].dt.strftime("%Y-%m") != df["order_month"])
    ),
    "check",
    ""
)

df["quarter_check"] = np.where(
    (
        df["order_date"].notna() &
        df["order_quarter"].notna() &
        (("Q" + df["order_date"].dt.quarter.astype("Int64").astype(str)) != df["order_quarter"])
    ),
    "check",
    ""
)

df["dow_check"] = np.where(
    (
        df["order_date"].notna() &
        df["day_of_week"].notna() &
        (df["order_date"].dt.day_name() != df["day_of_week"])
    ),
    "check",
    ""
)

In [25]:
df["month_check"] = np.where(
    df["order_date"].dt.strftime("%Y-%m") != df["order_month"],
    "check",
    ""
)

In [26]:
df["quarter_check"] = np.where(
    ("Q" + df["order_date"].dt.quarter.astype("Int64").astype(str)) != df["order_quarter"],
    "check",
    ""
)

In [27]:
df["dow_check"] = np.where(
    df["order_date"].dt.day_name() != df["day_of_week"],
    "check",
    ""
)

In [28]:
# Optional: flag suspicious discount values
df["discount_check"] = np.where(df["discount_pct"] > 0.5, "check", "")

In [29]:
# Final cleanup: overwrite totals with calculated values if you want consistency
df["order_total"] = df["order_total_calc"].round(2)
df["shipping_cost"] = df["shipping_cost"].round(2)

In [40]:
# Remove helper columns if you do not want them in final output
# Keep them if you want to audit the cleaning.
issues = df[
    (df["shipping_cost_check"] == "check") |
    (df["order_total_check"] == "check") |
    (df["year_check"] == "check") |
    (df["month_check"] == "check") |
    (df["quarter_check"] == "check") |
    (df["dow_check"] == "check") |
    (df["discount_check"] == "check")
].copy()

In [41]:
df = df.drop(columns=["discount_amount", "shipping_cost_calc", "order_total_calc"])

# Drop duplicates
df = df.drop_duplicates()

KeyError: "['discount_amount', 'shipping_cost_calc', 'order_total_calc'] not found in axis"

In [42]:
# Save outputs
df.to_csv(output_file, index=False)
issues.to_csv(issues_file, index=False)

print(f"Saved cleaned file: {output_file}")
print(f"Saved flagged rows: {issues_file}")

Saved cleaned file: orders_cleaned.csv
Saved flagged rows: orders_issues.csv
